# 🥉 Bronze — Ingesta Kafka → Delta Lake
**LogiLake | D'LOGIA** — Capa Bronze del Medallion Architecture

Este notebook consume mensajes del topic `olist_orders` en Kafka usando
**Spark Structured Streaming** y los persiste en Delta Lake como datos raw.

Prerrequisitos:
- Kafka corriendo local (o via ngrok al puerto 29092)
- Topic `olist_orders` creado (`bash kafka/topic_config.sh create`)
- Producer ejecutándose (`python kafka/olist_producer.py`)

In [ ]:
# ── Configuración de rutas (Databricks Community Edition) ────────────────────
# En Community Edition usamos DBFS (/dbfs/) como storage
CATALOG        = "logilake"   # prefijo lógico
BRONZE_PATH    = f"/FileStore/{CATALOG}/bronze/olist_orders"
CHECKPOINT_PATH= f"/FileStore/{CATALOG}/checkpoints/bronze_olist"

# Bootstrap server de Kafka
# Opción A — Local con ngrok:  'X.tcp.ngrok.io:XXXXX'
# Opción B — Misma red docker: 'host.docker.internal:29092'
KAFKA_BOOTSTRAP = "host.docker.internal:29092"  # AJUSTAR según tu entorno
KAFKA_TOPIC     = "olist_orders"

print(f"Bronze path:     {BRONZE_PATH}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Kafka broker:    {KAFKA_BOOTSTRAP}")

In [ ]:
# ── Instalar dependencias (solo primera vez en Community Edition) ─────────────
# El package kafka-clients viene incluido en Databricks Runtime 13+
# Solo es necesario si el cluster no lo tiene
%pip install kafka-python --quiet

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, ArrayType
import json

# Schema del payload Kafka (debe coincidir con olist_producer.py)
OLIST_SCHEMA = StructType([
    StructField("event_id",             StringType(),  True),
    StructField("order_id",             StringType(),  False),
    StructField("customer_id",          StringType(),  True),
    StructField("order_status",         StringType(),  True),
    StructField("order_purchase_ts",    StringType(),  True),
    StructField("order_approved_ts",    StringType(),  True),
    StructField("order_delivered_ts",   StringType(),  True),
    StructField("order_estimated_ts",   StringType(),  True),
    StructField("item_count",           IntegerType(), True),
    StructField("categories",           ArrayType(StringType()), True),
    StructField("seller_states",        ArrayType(StringType()), True),
    StructField("total_items_value",    FloatType(),   True),
    StructField("total_freight",        FloatType(),   True),
    StructField("payment_type",         StringType(),  True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value",        FloatType(),   True),
    StructField("review_score",         FloatType(),   True),
    StructField("ingested_at",          StringType(),  True),
    StructField("source",              StringType(),   True),
])

print("Schema definido:", len(OLIST_SCHEMA.fields), "campos")

In [ ]:
# ── Leer stream desde Kafka ───────────────────────────────────────────────────
kafka_stream_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")   # Cambia a 'latest' en producción
    .option("maxOffsetsPerTrigger", 10_000)  # Control de ingesta
    .option("failOnDataLoss", "false")
    .load()
)

print("Schema raw de Kafka:")
kafka_stream_raw.printSchema()

In [ ]:
# ── Deserializar JSON + añadir metadatos de Kafka ─────────────────────────────
bronze_stream = (
    kafka_stream_raw
    # El value de Kafka llega como bytes → cast a string
    .withColumn("value_str", F.col("value").cast("string"))
    # Parsear el JSON con el schema Olist
    .withColumn("data", F.from_json(F.col("value_str"), OLIST_SCHEMA))
    # Expandir campos anidados
    .select(
        "data.*",
        # Metadatos de Kafka (útiles para debugging y replay)
        F.col("topic").alias("kafka_topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        # Metadato de ingesta al bronze
        F.current_timestamp().alias("bronze_ingested_at"),
    )
    # Filtro mínimo: descartar mensajes sin order_id
    .filter(F.col("order_id").isNotNull())
)

print("Schema Bronze enriquecido:")
bronze_stream.printSchema()

In [ ]:
# ── Escribir a Delta Lake Bronze ──────────────────────────────────────────────
bronze_query = (
    bronze_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(processingTime="30 seconds")
    .start(BRONZE_PATH)
)

print(f"Stream iniciado. ID: {bronze_query.id}")
print(f"Estado: {bronze_query.status}")

In [ ]:
# ── Monitoreo del stream (ejecutar en celda separada) ─────────────────────────
# Esperar a que procese algunos batches
import time
time.sleep(35)

# Ver progreso
print("Estado del query:", bronze_query.status)
if bronze_query.recentProgress:
    last = bronze_query.recentProgress[-1]
    print(f"Rows procesados: {last.get('numInputRows', 0)}")
    print(f"Throughput:      {last.get('processedRowsPerSecond', 0):.1f} rows/s")
    print(f"Trigger:         {last.get('batchId', 'N/A')}")

In [ ]:
# ── Verificación: leer Bronze como batch ──────────────────────────────────────
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

print(f"Total registros en Bronze: {bronze_df.count():,}")
print(f"Particiones: {bronze_df.rdd.getNumPartitions()}")

# Muestra de datos
display(bronze_df.select(
    "order_id", "order_status", "order_purchase_ts",
    "item_count", "payment_value", "review_score",
    "kafka_partition", "kafka_offset", "bronze_ingested_at"
).limit(10))

# Distribución de estados
display(
    bronze_df.groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

In [ ]:
# ── Alternativa: Ingesta batch desde CSV (sin Kafka) ─────────────────────────
# Usar este bloque si Kafka no está disponible.
# Carga los CSVs directamente desde DBFS y escribe en Bronze.

RAW_DATA_PATH = "/FileStore/logilake/data/raw"

# Subir los CSV desde tu máquina:
# En Databricks: Data > Add Data > DBFS > Upload File

def load_olist_batch(raw_path: str) -> "DataFrame":
    orders   = spark.read.csv(f"{raw_path}/olist_orders_dataset.csv",         header=True, inferSchema=True)
    items    = spark.read.csv(f"{raw_path}/olist_order_items_dataset.csv",     header=True, inferSchema=True)
    payments = spark.read.csv(f"{raw_path}/olist_order_payments_dataset.csv",  header=True, inferSchema=True)
    reviews  = spark.read.csv(f"{raw_path}/olist_order_reviews_dataset.csv",   header=True, inferSchema=True)
    products = spark.read.csv(f"{raw_path}/olist_products_dataset.csv",        header=True, inferSchema=True)
    category = spark.read.csv(f"{raw_path}/product_category_name_translation.csv", header=True, inferSchema=True)
    sellers  = spark.read.csv(f"{raw_path}/olist_sellers_dataset.csv",         header=True, inferSchema=True)

    # Join incremental
    items_rich = items.join(products.join(category, "product_category_name", "left"),
                            "product_id", "left") \
                      .join(sellers.select("seller_id","seller_state","seller_city"), "seller_id", "left")

    items_agg = items_rich.groupBy("order_id").agg(
        F.count("order_item_id").alias("item_count"),
        F.sum("price").alias("total_items_value"),
        F.sum("freight_value").alias("total_freight"),
        F.collect_set("product_category_name_english").alias("categories"),
        F.collect_set("seller_state").alias("seller_states"),
    )

    payments_agg = payments.groupBy("order_id").agg(
        F.first("payment_type").alias("payment_type"),
        F.max("payment_installments").alias("payment_installments"),
        F.sum("payment_value").alias("payment_value"),
    )

    reviews_latest = reviews.orderBy("review_creation_date") \
                             .dropDuplicates(["order_id"]) \
                             .select("order_id", "review_score")

    return orders.join(items_agg, "order_id", "left") \
                 .join(payments_agg, "order_id", "left") \
                 .join(reviews_latest, "order_id", "left") \
                 .withColumn("bronze_ingested_at", F.current_timestamp()) \
                 .withColumn("source", F.lit("batch_csv_v1"))

# Descomentar para ejecutar:
# bronze_batch_df = load_olist_batch(RAW_DATA_PATH)
# bronze_batch_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(BRONZE_PATH)
# print(f"Bronze batch: {bronze_batch_df.count():,} filas")

In [ ]:
# ── Detener el stream cuando sea necesario ────────────────────────────────────
# bronze_query.stop()
# print("Stream detenido")